# EFT Limit Extraction with Toys

Demonstrates Wilson coefficient limits using asymptotic and toy-based hypothesis testing.

**Model:** $\sigma(c) = \sigma_{SM} \cdot (1 + Ac + Bc^2)$

**Architecture:** This notebook demonstrates the separation of concerns:
- **Test Statistics** (`QTilde`): Compute likelihood ratios
- **Distributions** (`QTildeAsymptotic`, `EmpiricalDistribution`): Convert q → p-values
- **Calculator** (`HypoTestCalculator`): Orchestrates the workflow
- **ToyGenerator**: Creates empirical distributions from Monte Carlo

In [1]:
import jax
import jax.numpy as jnp
from jax.scipy.special import gammaln

import everwillow.statelib as sl
from everwillow.inference.hypotest import (
    HypoTestCalculator,
    QTilde,
    QTildeAsymptotic,
    ToyGenerator,
    expected_upper_limit,
    upper_limit,
    upper_limit_toys,
)

jax.config.update("jax_enable_x64", True)

## Model Definition

In [ ]:
# EFT parameters
SM_PRED = 100.0  # SM prediction at c=0
A = 0.5  # Linear (interference) term
B = 0.1  # Quadratic term
ntoys = 10000  # Number of toys for toy-based limit


def nll_fn(params: dict, observation: dict) -> float:
    """NLL for single-bin Poisson measurement."""
    c = params["c"]
    pred = SM_PRED * (1.0 + c * A + c**2 * B)
    pred = jnp.maximum(pred, 1e-10)
    obs = observation["main"]
    return pred - obs * jnp.log(pred) + gammaln(obs + 1)


def predict_fn(params_state: sl.State) -> dict:
    """Expected observation given parameters (for Asimov)."""
    c = params_state.to_pytree()["c"]
    expected = SM_PRED * (1.0 + c * A + c**2 * B)
    return {"main": jnp.maximum(expected, 0.0)}


def sample_fn(params_state: sl.State, key) -> dict:
    """Sampling function for toy generation."""
    c = params_state.to_pytree()["c"]
    expected = SM_PRED * (1.0 + c * A + c**2 * B)
    return {"main": jax.random.poisson(key, jnp.maximum(expected, 0.0))}

## Setup

In [ ]:
# Observed data
observed = {"main": 100.0}

# Initial parameters
params = sl.State.from_pytree({"c": 0.0})

# Calculator and distributions (separation of concerns!)
calc = HypoTestCalculator(test_statistic=QTilde())
asymp_dist = QTildeAsymptotic()
toy_gen = ToyGenerator(test_statistic=QTilde(), ntoys=ntoys)
key = jax.random.key(42)

print(f"Model: sigma/sigma_SM = 1 + {A}c + {B}c^2")
print(
    f"Observed: {observed['main']:.0f} events | SM pred: {SM_PRED:.0f} | Toys: {ntoys}"
)

## CLs Comparison

In [ ]:
print("c\t Asymp\t  Toys")
print("-" * 25)
for i, c_test in enumerate([0.0, 0.1, 0.3, 0.5, 0.7, 1.0]):
    # Asymptotic: use QTildeAsymptotic distribution
    result_a = calc(
        nll_fn,
        params,
        observed,
        ("c",),
        c_test,
        distribution=asymp_dist,
        predict_fn=predict_fn,
    )

    # Toy-based: generate EmpiricalDistribution then use calculator
    emp_dist = toy_gen.generate(
        nll_fn,
        params,
        observed,
        ("c",),
        c_test,
        sample_fn=sample_fn,
        key=jax.random.fold_in(key, i),
    )
    result_t = calc(nll_fn, params, observed, ("c",), c_test, distribution=emp_dist)

    print(f"{c_test:.1f}\t {float(result_a.cl_s):.4f}\t {float(result_t.cl_s):.4f}")

## Upper Limits (95% CL)

In [ ]:
# Asymptotic limit
limit_asymp = upper_limit(
    lambda poi: calc(
        nll_fn,
        params,
        observed,
        ("c",),
        poi,
        distribution=asymp_dist,
        predict_fn=predict_fn,
    ).palt,
    bounds=(0.0, 2.0),
    level=0.05,
)
print(f"Asymptotic: c < {float(limit_asymp):.3f}")

In [ ]:
# Toy-based limit
limit_toys = upper_limit_toys(
    lambda poi, k: calc(
        nll_fn,
        params,
        observed,
        ("c",),
        poi,
        distribution=toy_gen.generate(
            nll_fn,
            params,
            observed,
            ("c",),
            poi,
            sample_fn=sample_fn,
            key=k,
        ),
    ).palt,
    bounds=(0.0, 2.0),
    key=key,
    level=0.05,
)
print(f"Toys:       c < {float(limit_toys):.3f}")

## Expected Limit Bands

Expected limits under background-only hypothesis (Brazil bands).

In [ ]:
# Compute all limits with Brazil bands in one call
limits = expected_upper_limit(
    lambda poi: calc(
        nll_fn,
        params,
        observed,
        ("c",),
        poi,
        distribution=asymp_dist,
        predict_fn=predict_fn,
    ),
    bounds=(-100.0, 100.0),
)

print("95% CL Upper Limits (Brazil bands):")
print(f"  -2sigma: c < {float(limits.minus_2sigma):.3f}")
print(f"  -1sigma: c < {float(limits.minus_1sigma):.3f}")
print(f"  Exp:     c < {float(limits.expected):.3f}")
print(f"  +1sigma: c < {float(limits.plus_1sigma):.3f}")
print(f"  +2sigma: c < {float(limits.plus_2sigma):.3f}")
print(f"  Obs:     c < {float(limits.observed):.3f}")

## JIT Compilation Timing

In [ ]:
import time


@jax.jit
def compute_limit_jit(key):
    return upper_limit_toys(
        lambda poi, k: calc(
            nll_fn,
            params,
            observed,
            ("c",),
            poi,
            distribution=toy_gen.generate(
                nll_fn,
                params,
                observed,
                ("c",),
                poi,
                sample_fn=sample_fn,
                key=k,
            ),
        ).palt,
        bounds=(0.0, 2.0),
        key=key,
        level=0.05,
    )


# First call includes compilation
start = time.time()
limit1 = compute_limit_jit(jax.random.key(1))
t_compile = time.time() - start

# Second call uses cached compilation
start = time.time()
limit2 = compute_limit_jit(jax.random.key(1))
t_cached = time.time() - start

print(f"First call (compile): {t_compile:.2f}s → c < {float(limit1):.3f}")
print(f"Second call (cached): {t_cached:.2f}s → c < {float(limit2):.3f}")
print(f"Speedup: {t_compile / t_cached:.1f}x")